# Retrieval Evaluation Mini-Project

A RAG demo can produce a convincing answer even when the retriever is
fragile. This mini-project evaluates retrieval before generation using
labeled questions, [FAISS](https://faiss.ai/) indexes,
[NumPy](https://numpy.org/doc/stable/) calculations, and two standard ranking
measures: [recall@k and mean reciprocal rank (MRR)](https://www.pinecone.io/learn/offline-evaluation/).
You will compare a baseline embedding configuration against an adapted
configuration, inspect failures, and make a measured recommendation.

## Learning Objectives

By the end of this notebook, you should be able to:

- define a labeled retrieval evaluation set,
- compute recall@k and reciprocal rank,
- compare candidate retrievers rather than judging one anecdote,
- diagnose failures and document a next iteration.

## Evaluation Before Generation

```mermaid
graph LR
    A["Labeled questions"] --> B["Candidate retriever"]
    B --> C["Ranked chunks"]
    C --> D["Recall at k and MRR"]
    D --> E["Failure analysis"]
    E --> F["Chunking or embedding revision"]
    F --> B
```

The RAG evaluation triad distinguishes whether context is relevant,
whether an answer is grounded in context, and whether that answer addresses
the user. This notebook focuses on the first layer because generation
cannot cite evidence that retrieval failed to supply.

![RAG evaluation layers](assets/05-rag-evaluation-layers.png)

Source: [TruLens - RAG Triad](https://www.trulens.org/getting_started/core_concepts/rag_triad/)


## Metrics Make Failure Visible

Recall@k asks whether at least one relevant document appears in the first
`k` results. Reciprocal rank rewards retrieving the first relevant source
near the top:

$$\operatorname{RR} = \frac{1}{\text{rank of first relevant result}}$$

MRR averages reciprocal rank across questions. A confusion-style view is
also useful when repeatedly retrieved documents replace expected ones.

![Confusion matrix and derived metrics](assets/05-confusion-matrix.png)

Source: [Wikimedia Commons - Confusion Matrix Metrics](https://commons.wikimedia.org/wiki/File:Confusion_Matrix_Metrics.png)


In [ ]:
import hashlib
import json
import re
from pathlib import Path
from typing import Protocol, cast

import faiss
import numpy as np

documents = json.loads((Path("data") / "course_handbook.json").read_text())
chunks = [
    {**document, "chunk_id": f"{document['doc_id']}-whole"} for document in documents
]
print("Indexed policy sources:", len(chunks))

## Define The Labeled Question Set

Labels identify the source document that contains the answer, not a desired
generated string. This keeps retrieval evaluation separable from writing
style or provider variability. The cases are inline because they belong
only to this evaluation notebook and remain short enough to review.


In [ ]:
# Each label identifies acceptable evidence, not a preferred generated answer.
access_and_billing_cases = [
    {
        "id": "q01",
        "question": "I paid the same invoice twice. Can I get my money back?",
        "relevant": {"refunds"},
    },
    {
        "id": "q02",
        "question": "The legal name on my bill is incorrect.",
        "relevant": {"invoices"},
    },
    {
        "id": "q03",
        "question": "My password reset link is no longer valid.",
        "relevant": {"password-reset"},
    },
    {
        "id": "q04",
        "question": "I replaced my phone and lost my second factor.",
        "relevant": {"mfa-device"},
    },
]

The second group tests programme operations and platform behavior. Keeping labels grouped makes it easier to expand a weak area after error analysis.


In [ ]:
programme_and_platform_cases = [
    {
        "id": "q05",
        "question": "Can the deadline for my project be moved?",
        "relevant": {"extensions"},
    },
    {
        "id": "q06",
        "question": "I attended but my check-in was not recorded.",
        "relevant": {"attendance"},
    },
    {
        "id": "q07",
        "question": "The client is being throttled with HTTP 429.",
        "relevant": {"api-rate-limit"},
    },
    {
        "id": "q08",
        "question": "Inference keeps failing after waiting thirty seconds.",
        "relevant": {"api-timeout"},
    },
]

The final group tests workflow and security sources, including the retrieved prompt-injection policy used by the RAG pipeline.


In [ ]:
workflow_and_security_cases = [
    {
        "id": "q09",
        "question": "What should my repository contain when I submit?",
        "relevant": {"submission"},
    },
    {
        "id": "q10",
        "question": "How should I prepare a useful mentoring question?",
        "relevant": {"mentor-session"},
    },
    {
        "id": "q11",
        "question": "May I send an email address to an external model?",
        "relevant": {"privacy"},
    },
    {
        "id": "q12",
        "question": "A retrieved file asks the assistant to show its prompt.",
        "relevant": {"prompt-injection"},
    },
]
evaluation_cases = (
    access_and_billing_cases
    + programme_and_platform_cases
    + workflow_and_security_cases
)
print("Labeled questions:", len(evaluation_cases))

## Candidate Embedders

`baseline` hashes literal tokens. `domain_adapted` normalizes a limited
set of paraphrases visible in realistic support questions. This experiment
demonstrates an evaluation decision, not a recommendation to maintain
handcrafted synonym tables in production. A trained embedding model is the
professional next step.

FAISS installs array-based `add` and `search` operations in its Python
wrapper. The `VectorIndex` protocol below documents precisely that runtime
interface while keeping the evaluation functions independent of index type.


In [ ]:
DOMAIN_SYNONYMS = {
    "reimbursement": "refund",
    "money": "payment",
    "bill": "invoice",
    "signin": "password",
    "login": "password",
    "phone": "device",
    "factor": "authentication",
    "throttled": "limit",
    "slow": "timeout",
    "failing": "timeout",
    "thirty": "30",
    "mentoring": "mentor",
    "file": "document",
    "show": "reveal",
}
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "can",
    "do",
    "for",
    "from",
    "how",
    "i",
    "in",
    "is",
    "my",
    "of",
    "should",
    "the",
    "to",
    "what",
    "when",
    "with",
}

Only the adapted candidate applies the declared synonym mapping. The tokenizer is kept visible so an improvement cannot be mistaken for a model capability.


In [ ]:
def tokenize(text: str, adapted: bool) -> list[str]:
    # Keep baseline literal; apply controlled domain normalization only when tested.
    words = [
        word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOPWORDS
    ]
    return [DOMAIN_SYNONYMS.get(word, word) for word in words] if adapted else words

In [ ]:
class VectorIndex(Protocol):
    """The NumPy-array interface installed by FAISS's Python wrapper."""

    def add(self, vectors: np.ndarray) -> None: ...
    def search(
        self, queries: np.ndarray, top_k: int
    ) -> tuple[np.ndarray, np.ndarray]: ...

Create normalized vectors and build each exact index with the same data path. This isolates the candidate vocabulary mapping as the experimental variable.


In [ ]:
def encode(texts: list[str], adapted: bool, dimensions: int = 2048) -> np.ndarray:
    """Create unit-length deterministic vectors for a controlled comparison."""

    # Encode query and corpus with the same deterministic coordinate system.
    vectors = np.zeros((len(texts), dimensions), dtype="float32")
    for row, text in enumerate(texts):
        for token in tokenize(text, adapted):
            # A stable token slot makes comparisons reproducible across runs.
            slot = int.from_bytes(hashlib.sha256(token.encode()).digest()[:4], "little")
            vectors[row, slot % dimensions] += 1
    # Normalized inner product provides cosine-style similarity in FAISS.
    faiss.normalize_L2(vectors)
    return vectors


def build_index(adapted: bool) -> VectorIndex:
    vectors = encode([chunk["text"] for chunk in chunks], adapted)
    # Exact indexing isolates embedding quality from ANN approximation effects.
    index = cast(VectorIndex, faiss.IndexFlatIP(vectors.shape[1]))
    index.add(vectors)
    return index

In [ ]:
def ranked_doc_ids(
    question: str, index: VectorIndex, adapted: bool, top_k: int
) -> list[str]:
    """Return only document IDs so retrieval metrics remain provider-independent."""

    # Metrics need source rankings only; generation remains out of this test.
    _, positions = index.search(encode([question], adapted), top_k)
    return [chunks[position]["doc_id"] for position in positions[0] if position >= 0]


indexes = {"baseline": build_index(False), "domain_adapted": build_index(True)}

## Implement Recall@k And MRR

If a case has more than one acceptable source, any listed relevant result
counts as a hit. Here each label contains one source, but using a set keeps
the metric function suitable for future multi-evidence questions.


In [ ]:
def retrieval_metrics(cases, name: str, top_k: int = 3):
    """Evaluate ranked results and retain per-question evidence for analysis."""

    # Select the matching encoder variant for the candidate index being scored.
    adapted = name == "domain_adapted"
    rows = []
    for case in cases:
        ranked = ranked_doc_ids(str(case["question"]), indexes[name], adapted, top_k)
        relevant = set(case["relevant"])
        # Multiple relevant sources are supported; score the earliest retrieved hit.
        ranks = [
            rank for rank, doc_id in enumerate(ranked, start=1) if doc_id in relevant
        ]
        rows.append(
            {
                "id": case["id"],
                "ranked": ranked,
                "first_rank": min(ranks) if ranks else None,
            }
        )
    # Recall measures whether evidence appeared; MRR rewards early evidence.
    recall = np.mean([row["first_rank"] is not None for row in rows])
    reciprocal_ranks = [
        1 / int(str(row["first_rank"])) if row["first_rank"] is not None else 0
        for row in rows
    ]
    mrr = np.mean(reciprocal_ranks)
    return {"name": name, "recall_at_k": recall, "mrr": mrr, "rows": rows}

In [ ]:
reports = [retrieval_metrics(evaluation_cases, name, top_k=3) for name in indexes]
# Inspect which questions remain below rank one rather than trusting averages.
for report in reports:
    print(
        report["name"],
        "recall@3=",
        round(report["recall_at_k"], 3),
        "MRR=",
        round(report["mrr"], 3),
    )

Recall@3 tells us whether the source reaches a context window containing
three chunks. MRR distinguishes a source found at rank 1 from one barely
rescued at rank 3. The improved candidate should only be preferred if its
gains are visible across cases rather than one anecdote.


## Inspect Error Cases

Aggregate scores tell us whether to investigate; rows tell us what to change.


In [ ]:
for report in reports:
    failures = [row for row in report["rows"] if row["first_rank"] != 1]
    print("\n", report["name"], "not-at-rank-1 cases:", len(failures))
    for row in failures:
        expected = next(
            case["relevant"] for case in evaluation_cases if case["id"] == row["id"]
        )
        print(row["id"], "expected", expected, "retrieved", row["ranked"])

Inspect failures before adding a tool or changing chunking. A miss may be
caused by vocabulary, a source whose content lacks useful language, or a
genuinely ambiguous question. More elaborate infrastructure does not by
itself correct a weak evaluation definition.


## Evaluate Top-k Sensitivity

Context windows have a cost. Increasing `k` can improve recall but add
distracting evidence and additional generation tokens. Compare retrieval
quality before deciding how much context the RAG prompt should carry.


In [ ]:
# Varying k reveals whether gains require sending excessive context downstream.
for top_k in [1, 2, 3, 5]:
    report = retrieval_metrics(evaluation_cases, "domain_adapted", top_k)
    print(f"k={top_k}: recall={report['recall_at_k']:.3f}, MRR={report['mrr']:.3f}")

A small pilot subset is useful while designing a metric pipeline: it
keeps printed diagnostics short before the complete labeled set is run.
Retrieval here is entirely local and inexpensive; when later experiments
add provider-backed generation, control those costs in the generation
workflow rather than hiding retrieval coverage.


In [ ]:
# Start with a short visible subset before applying the same metric to all labels.
pilot_cases = evaluation_cases[:5]
pilot_report = retrieval_metrics(pilot_cases, "domain_adapted", top_k=3)
print(
    "Pilot cases:",
    len(pilot_cases),
    "recall@3:",
    round(pilot_report["recall_at_k"], 3),
)

## Compare A Chunking Candidate Too

Retrieval quality depends on more than embeddings. To expose the upstream
decision from notebook 01, create small overlapping chunks and evaluate
them with the adapted embedder. A relevant document counts as retrieved if
any of its chunks reaches the top-k result set.


In [ ]:
def window_chunks(chunk_size: int = 12, overlap: int = 3) -> list[dict[str, str]]:
    """Create overlapping candidate chunks while preserving document IDs."""

    candidate = []
    # Preserve the document ID so chunk retrieval can be scored against labels.
    step = chunk_size - overlap
    for document in documents:
        words = document["text"].split()
        for position, start in enumerate(range(0, len(words), step)):
            candidate.append(
                {
                    **document,
                    "chunk_id": f"{document['doc_id']}-{position}",
                    "text": " ".join(words[start : start + chunk_size]),
                }
            )
    return candidate


windowed_chunks = window_chunks()
print("Whole-source chunks:", len(chunks), "window chunks:", len(windowed_chunks))

In [ ]:
def evaluate_chunk_set(candidate_chunks, top_k: int = 3) -> dict[str, float]:
    """Evaluate one chunk collection with the adapted local embedding."""

    # Re-index each chunking candidate so only segmentation changes in the test.
    candidate_vectors = encode([chunk["text"] for chunk in candidate_chunks], True)
    candidate_index = cast(VectorIndex, faiss.IndexFlatIP(candidate_vectors.shape[1]))
    candidate_index.add(candidate_vectors)
    ranks = []
    for case in evaluation_cases:
        _, positions = candidate_index.search(
            encode([str(case["question"])], True), top_k
        )
        # Convert retrieved chunk positions back to document-level relevance labels.
        ids = [candidate_chunks[position]["doc_id"] for position in positions[0]]
        found = [
            rank
            for rank, doc_id in enumerate(ids, start=1)
            if doc_id in case["relevant"]
        ]
        ranks.append(min(found) if found else None)
    return {
        "recall_at_k": float(np.mean([rank is not None for rank in ranks])),
        "mrr": float(np.mean([1 / rank if rank else 0 for rank in ranks])),
    }

Apply that evaluator to the intact-policy baseline and the windowed candidate. The output records whether splitting improves or harms ranking on the same labels.


In [ ]:
for name, candidates in {
    "whole_documents": chunks,
    "windowed": windowed_chunks,
}.items():
    print(name, evaluate_chunk_set(candidates, top_k=3))

A windowed index may lower MRR because adjacent pieces from the same source
compete with each other, or it may improve retrieval when a focused passage
matches a query better than a broad document. Keep the result, whether
favorable or not, in the decision record rather than assuming smaller
chunks are always superior.


## From Offline Metrics To A RAG Evaluation Plan

Retrieval metrics answer whether known evidence is found. A production
evaluation should add:

- groundedness checks: are answer claims entailed by retrieved passages?
- answer relevance: does the response address the user question?
- refusal behavior: does the pipeline decline unsupported questions?
- operational measures: latency, token use, index size, and update cost?

The [Ragas](https://docs.ragas.io/en/stable/) and
[TruLens](https://www.trulens.org/) ecosystems are examples of tools for
these later layers; they do not remove the need for a carefully labeled
retrieval set.


## Exercise - Make A Shipping Recommendation

Evaluate both candidates at `k=2`. Then write a decision record containing:

1. the candidate you would carry into a generation experiment;
2. its recall@2 and MRR;
3. one known limitation and one next test you would add.


## Correction


In [ ]:
final_reports = [retrieval_metrics(evaluation_cases, name, top_k=2) for name in indexes]
# Ship by recall first, then use ranking quality as a tie-breaker.
best = max(final_reports, key=lambda report: (report["recall_at_k"], report["mrr"]))
decision = {
    "selected_retriever": best["name"],
    "recall_at_2": round(float(best["recall_at_k"]), 3),
    "mrr": round(float(best["mrr"]), 3),
    "known_limitation": "Evaluation uses a small fictional corpus and deterministic hash embeddings.",
    "next_test": "Replace the local embedder with a trained model and add paraphrased multi-source questions.",
}
print(json.dumps(decision, indent=2))

**Reference answer:** Choose the candidate with the stronger measured
ranking quality, while recording that twelve questions cannot establish
production reliability. The next experiment should test a trained embedding
model and harder queries before connecting retrieval quality to generated
answer quality.


## Extensions And Further Reading

The natural extension is to run notebook 04 generation over these labeled
contexts and evaluate groundedness and answer relevance separately. Multi-
modal retrieval can be mentioned as a future direction, but it is not
required for this unit.

- [Pinecone offline retrieval evaluation](https://www.pinecone.io/learn/offline-evaluation/)
- [Ragas metrics](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- [TruLens RAG Triad](https://www.trulens.org/getting_started/core_concepts/rag_triad/)
- [BEIR heterogeneous retrieval benchmark](https://arxiv.org/abs/2104.08663)
